# Kaggle submission (OFFLINE) — Akkadian → English (ByT5)

Code-competition без интернета: Kaggle приватно перезапускает выбранную версию со
скрытым тестом и забирает `submission.csv` из Output. Интернет **выключен**, поэтому
ноутбук самодостаточный: код инференса вшит ниже, модель грузится из подключённого
Kaggle Dataset (см. `kaggle_export_model.ipynb`), наш пакет/HF Hub не нужны.

**Перед сабмитом:**
1. **Input → + Add Input**: соревнование *Deep Past Initiative* и датасет с моделью (`akkadian-models`).
2. **Internet → Off** (требование соревнования).
3. **Accelerator → GPU T4** (НЕ P100 — несовместим с текущим PyTorch). На CPU тоже
   отработает, просто медленнее.
4. Проверь `MODEL_DIR` и `NORMALIZE`, Save & Run All → `submission.csv` в Output → **Submit**.

In [ ]:
# путь к папке модели внутри подключённого датасета
MODEL_DIR = "/kaggle/input/akkadian-models/byt5-baseline-docs-raw"
# NORMALIZE должен совпадать с обучением: baseline(src_raw)->False, exp1+(нормализ.)->True
NORMALIZE = False
NUM_BEAMS = 4
TEST = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
OUT = "/kaggle/working/submission.csv"

In [ ]:
# --- вшитая нормализация (копия akkadian_nmt/normalize.py) ---
import re, unicodedata
_SUB = str.maketrans("₀₁₂₃₄₅₆₇₈₉ₓ", "0123456789x")
_DAMAGE = re.compile(r"[⸢⸣\[\]!?#*]")
_ANGLE = re.compile(r"<+[^<>]*>+")
_WS = re.compile(r"\s+")
GAP = "…"

def normalize_translit(text, keep_damage_marks=False):
    text = unicodedata.normalize("NFC", text).translate(_SUB)
    text = _ANGLE.sub(GAP, text)
    if not keep_damage_marks:
        text = _DAMAGE.sub("", text)
    text = re.sub(rf"(?:{GAP}\s*)+", GAP + " ", text)
    return _WS.sub(" ", text).strip()

In [ ]:
import torch, pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device).eval()
print("loaded", MODEL_DIR, "on", device)

In [ ]:
@torch.inference_mode()
def translate(texts, num_beams=4, batch_size=8, max_src=512, max_new=512, normalize=True):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        if normalize:
            batch = [normalize_translit(t) for t in batch]
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=max_src).to(device)
        gen = model.generate(**enc, num_beams=num_beams, max_new_tokens=max_new)
        out.extend(tok.batch_decode(gen, skip_special_tokens=True))
    return out

df = pd.read_csv(TEST)
hyps = translate(df["transliteration"].fillna("").tolist(),
                 num_beams=NUM_BEAMS, normalize=NORMALIZE)
pd.DataFrame({"id": df["id"], "translation": hyps}).to_csv(OUT, index=False)
print("wrote", OUT, "| rows:", len(hyps))
pd.read_csv(OUT).head()